In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_course_text_embeddings.npy
/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_embeddings_course_index.json
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json
/kaggle/input/datasets/ngocccchannn/llm-embeddings/llm_profile_embeddings.npy
/kaggle/input/datasets/ngocccchannn/llm-embeddings/full_3769_profiles.json


In [2]:
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import subprocess
import sys 

try:
    import faiss
    print("[Info] FAISS đã được cài đặt.")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "faiss-cpu"])
    import faiss

coldrec_path = "/kaggle/working/ColdRec"
if not os.path.isdir(coldrec_path):
    subprocess.check_call(["git", "clone", "https://github.com/YuanchenBei/ColdRec.git", coldrec_path])

# Vá tệp model/__init__.py để loại bỏ module SimCSR lỗi
init_file_path = os.path.join(coldrec_path, "model", "__init__.py")
if os.path.exists(init_file_path):
    with open(init_file_path, "r", encoding="utf-8") as f:
        init_content = f.read()
    if "SimCSR" in init_content:
        new_content = "\n".join([line for line in init_content.splitlines() if "SimCSR" not in line])
        with open(init_file_path, "w", encoding="utf-8") as f:
            f.write(new_content)
        print("[Info] Đã tự động vá tệp model/__init__.py.")

if coldrec_path not in sys.path:
    sys.path.insert(0, coldrec_path)

from model.DropoutNet import DropoutNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Info] Thiết bị tính toán: {device}")
print("[Info] PyTorch:", torch.__version__)
print("[Info] CUDA khả dụng:", torch.cuda.is_available())

def get_vram_usage():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 3)
    return 0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.3 MB/s eta 0:00:00


Cloning into '/kaggle/working/ColdRec'...


[Info] Đã tự động vá tệp model/__init__.py.
[Info] Thiết bị tính toán: cuda
[Info] PyTorch: 2.11.0+cu128
[Info] CUDA khả dụng: True


In [3]:
# [CELL 3 - NẠP DỮ LIỆU KỊCH BẢN COLD-ITEM VÀ XÂY DỰNG INDEX CHUẨN XÁC]
USE_LLM_PROFILE = False  # Đặt True để dùng embedding hồ sơ LLM, False để dùng văn bản gốc (raw_course_text)

embedding_dir = "/kaggle/input/datasets/ngocccchannn/raw-embeddings"
llm_dir = "/kaggle/input/datasets/ngocccchannn/llm-embeddings"

with open(f"{embedding_dir}/raw_embeddings_course_index.json", "r", encoding="utf-8") as f:
    course_index = json.load(f)

raw_item_texts = np.load(f"{embedding_dir}/raw_course_text_embeddings.npy") # [3769, 384]
if USE_LLM_PROFILE and os.path.exists(f"{llm_dir}/llm_profile_embeddings.npy"):
    item_texts = np.load(f"{llm_dir}/llm_profile_embeddings.npy") # [3769, 384] từ nhánh LLM
    print("[Info] Đang sử dụng đầu vào: LLM Pedagogical Profile Embeddings")
else:
    item_texts = raw_item_texts
    print("[Info] Đang sử dụng đầu vào: Raw Course Text Embeddings")

sem_tensor = torch.tensor(item_texts, dtype=torch.float32).to(device)

scenario_dir = "/kaggle/input/datasets/ngocccchannn/cold-scenario"
with open(f"{scenario_dir}/cold_items_val.json", "r", encoding="utf-8") as f:
    cold_val_ids = json.load(f)
with open(f"{scenario_dir}/cold_items_test.json", "r", encoding="utf-8") as f:
    cold_test_ids = json.load(f)

data_dir = "/kaggle/input/datasets/ngocccchannn/cold-scenario-data"
tr_cold = pd.read_parquet(f"{data_dir}/train_cold.parquet")
va_cold = pd.read_parquet(f"{data_dir}/val_cold.parquet")
te_cold = pd.read_parquet(f"{data_dir}/test_cold.parquet")

all_users = pd.concat([tr_cold['user_id'], va_cold['user_id'], te_cold['user_id']]).unique()
user_to_idx = {u: i for i, u in enumerate(all_users)}

tr_cold['user_id'] = tr_cold['user_id'].map(user_to_idx)
va_cold['user_id'] = va_cold['user_id'].map(user_to_idx)
te_cold['user_id'] = te_cold['user_id'].map(user_to_idx)

course_to_idx = {}
if 'course_id' in course_index and len(course_index['course_id']) == len(item_texts):
    course_to_idx = {str(c): i for i, c in enumerate(course_index['course_id'])}
    tr_cold['course_id'] = tr_cold['course_id'].map(lambda x: course_to_idx.get(str(x), int(x) if str(x).isdigit() else x))
    va_cold['course_id'] = va_cold['course_id'].map(lambda x: course_to_idx.get(str(x), int(x) if str(x).isdigit() else x))
    te_cold['course_id'] = te_cold['course_id'].map(lambda x: course_to_idx.get(str(x), int(x) if str(x).isdigit() else x))

tr_cold['course_id'] = tr_cold['course_id'].astype(int)
va_cold['course_id'] = va_cold['course_id'].astype(int)
te_cold['course_id'] = te_cold['course_id'].astype(int)

n_users = len(user_to_idx)
n_items = int(item_texts.shape[0])

target_te = te_cold.groupby('user_id')['course_id'].apply(set).to_dict()
target_va = va_cold.groupby('user_id')['course_id'].apply(set).to_dict()
mask_tr = tr_cold.groupby('user_id')['course_id'].apply(set).to_dict()
mask_va = va_cold.groupby('user_id')['course_id'].apply(set).to_dict() # Chỉ dùng để che khi đánh giá trên tập Test

# Tập ứng viên (a): Chỉ trên các khóa lạnh validation (251 khóa) và test (502 khóa)
def map_to_indices(id_list, c_to_idx):
    res = []
    for c in id_list:
        if str(c) in c_to_idx:
            res.append(c_to_idx[str(c)])
        elif str(c).isdigit():
            res.append(int(c))
    return res

cand_list_va = map_to_indices(cold_val_ids, course_to_idx)
cand_list_te = map_to_indices(cold_test_ids, course_to_idx)
if len(cand_list_va) == 0: cand_list_va = va_cold['course_id'].unique().tolist()
if len(cand_list_te) == 0: cand_list_te = te_cold['course_id'].unique().tolist()

# Tập ứng viên (b): Toàn bộ danh mục khóa học (All Items)
cand_list_b = list(range(n_items))

# Tách biệt tập khóa học lạnh (Ic) và tập khóa học ấm (Iw) theo đúng Mô tả ý tưởng
cold_item_set = set(cand_list_va) | set(cand_list_te)
warm_item_list = sorted(list(set(range(n_items)) - cold_item_set))

# Xây dựng nhóm trùng lặp dup_group_id theo Phụ lục A.2 trong Review của cô GVHD để phục vụ P-RQ3
profiles_path = f"{llm_dir}/full_3769_profiles.json"
dup_group_id_map = {i: i for i in range(n_items)}
if os.path.exists(profiles_path):
    with open(profiles_path, "r", encoding="utf-8") as f:
        prof_data = json.load(f)
    rows = prof_data if isinstance(prof_data, list) else prof_data.get("profiles", [])
    parent = {i: i for i in range(n_items)}
    def _find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    name_groups, about_groups = {}, {}
    for idx_r, r in enumerate(rows):
        cid = course_to_idx.get(str(r.get("course_id", "")), idx_r)
        if cid >= n_items: continue
        nm = str(r.get("course_name", r.get("name", ""))).strip().lower()
        ab = str(r.get("course_about", r.get("about", ""))).strip().lower()
        if nm: name_groups.setdefault(nm, []).append(cid)
        if ab: about_groups.setdefault(ab, []).append(cid)
    for grp in list(name_groups.values()) + list(about_groups.values()):
        for x in grp[1:]:
            parent[_find(x)] = _find(grp[0])
    dup_group_id_map = {i: _find(i) for i in range(n_items)}
else:
    # Fallback: phát hiện trùng văn bản gốc qua cosine similarity = 1.0 trên raw_course_text_embeddings
    raw_norm = raw_item_texts / (np.linalg.norm(raw_item_texts, axis=1, keepdims=True) + 1e-12)
    sim_mat = raw_norm @ raw_norm.T
    parent = {i: i for i in range(n_items)}
    def _find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    dup_pairs = np.argwhere(sim_mat > 0.9999)
    for i_p, j_p in dup_pairs:
        if i_p < j_p:
            parent[_find(int(j_p))] = _find(int(i_p))
    dup_group_id_map = {i: _find(i) for i in range(n_items)}

warm_dup_groups = {dup_group_id_map[i] for i in warm_item_list}
cold_test_dup_items = {i for i in cand_list_te if dup_group_id_map[i] in warm_dup_groups}
cold_test_genuine_items = set(cand_list_te) - cold_test_dup_items

print(f"[Info] Số lượng User: {n_users}, Số lượng Item: {n_items} (Warm Iw: {len(warm_item_list)}, Cold Ic: {len(cold_item_set)})")
print(f"[Info] Số lượng ứng viên cold-val (a): {len(cand_list_va)} | cold-test (a): {len(cand_list_te)}")
print(f"[Info] Phân nhóm P-RQ3 trên cold-test: Trùng khóa ấm = {len(cold_test_dup_items)} khóa | Mới hoàn toàn = {len(cold_test_genuine_items)} khóa")

[Info] Đang sử dụng đầu vào: Raw Course Text Embeddings
[Info] Số lượng User: 98220, Số lượng Item: 3769 (Warm Iw: 3016, Cold Ic: 753)
[Info] Số lượng ứng viên cold-val (a): 251 | cold-test (a): 502
[Info] Phân nhóm P-RQ3 trên cold-test: Trùng khóa ấm = 73 khóa | Mới hoàn toàn = 429 khóa


In [4]:
def topk_random_ties(sc, K, g):
    """sc: [B, N] điểm, các khóa bị loại có giá trị -inf. Trả về chỉ số top-K [B, K]."""
    perm = torch.argsort(torch.rand(sc.shape, generator=g), dim=1)   # hoán vị riêng từng hàng
    sp = torch.gather(sc, 1, perm)
    order = torch.sort(sp, dim=1, descending=True, stable=True).indices[:, :K]
    return torch.gather(perm, 1, order).numpy()

@torch.no_grad() 
def evaluate(scores_fn, target, mask, ks=(5, 10, 20), cand=None, bs=1024, tie_seed=0): 
    users = sorted(target); K = max(ks) 
    out = {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks} 
    g = torch.Generator().manual_seed(tie_seed) 
    disc = 1.0 / np.log2(np.arange(2, K + 2)) 
    
    for s in range(0, len(users), bs): 
        ub = users[s:s + bs]; 
        sc = scores_fn(torch.tensor(ub)).float().cpu() 
        if cand is not None: 
            sc[:, ~torch.as_tensor(cand)] = -float("inf") 
        for r, u in enumerate(ub): 
            if mask.get(u): sc[r, list(mask[u])] = -float("inf") 
        
        # Phá hòa ngẫu nhiên riêng biệt cho từng học viên
        top = topk_random_ties(sc, K, g)
        
        for r, u in enumerate(ub): 
            gt = target[u]; hit = np.isin(top[r], list(gt)).astype(float) 
            for k in ks: 
                h = hit[:k]; idcg = disc[:min(len(gt), k)].sum() 
                out[f"Recall@{k}"].append(h.sum() / len(gt)) 
                out[f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg) 
                out[f"HR@{k}"].append(float(h.sum() > 0)) 
    return {m: np.array(v) for m, v in out.items()} 

In [5]:
def topk_random_ties(sc, K, g):
    perm = torch.argsort(torch.rand(sc.shape, generator=g), dim=1)
    sp = torch.gather(sc, 1, perm)
    order = torch.sort(sp, dim=1, descending=True, stable=True).indices[:, :K]
    return torch.gather(perm, 1, order).numpy()

@torch.no_grad() 
def evaluate_scenarios(scores_fn, target, mask_train, mask_val=None, ks=(5, 10, 20), cand=None, bs=1024, tie_seed=0, item_subset=None): 
    users = sorted(target); K = max(ks) 
    out = {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks} 
    user_metrics = {} # Lưu chi tiết từng user để phục vụ bootstrap/Wilcoxon sau này
    g = torch.Generator().manual_seed(tie_seed) 
    disc = 1.0 / np.log2(np.arange(2, K + 2)) 
    
    for s in range(0, len(users), bs): 
        ub = users[s:s + bs]; sc = scores_fn(torch.tensor(ub)).float().cpu() 
        
        # Sửa lỗi che ứng viên bằng mặt nạ boolean chính xác tuyệt đối
        if cand is not None: 
            cand_mask = torch.ones(sc.shape[1], dtype=torch.bool)
            cand_mask[torch.as_tensor(cand)] = False
            sc[:, cand_mask] = -float("inf") 
            
        for r, u in enumerate(ub): 
            if mask_train.get(u): sc[r, list(mask_train[u])] = -float("inf") 
            if mask_val and mask_val.get(u): sc[r, list(mask_val[u])] = -float("inf")
        
        top = topk_random_ties(sc, K, g)
        
        for r, u in enumerate(ub): 
            gt = target[u] if item_subset is None else (target[u] & item_subset)
            if len(gt) == 0:
                continue
            hit = np.isin(top[r], list(gt)).astype(float) 
            u_res = {}
            for k in ks: 
                h = hit[:k]; idcg = disc[:min(len(gt), k)].sum() 
                rec = h.sum() / len(gt) if len(gt) > 0 else 0.0
                ndcg = (h * disc[:k]).sum() / idcg if idcg > 0 else 0.0
                hr = float(h.sum() > 0)
                
                out[f"Recall@{k}"].append(rec) 
                out[f"NDCG@{k}"].append(ndcg) 
                out[f"HR@{k}"].append(hr)
                
                u_res[f"Recall@{k}"] = rec
                u_res[f"NDCG@{k}"] = ndcg
                u_res[f"HR@{k}"] = hr
            user_metrics[u] = u_res
            
    return {m: np.array(v) for m, v in out.items()}, user_metrics

In [6]:
# [CELL 5 - TẠO VÀ NẠP TỆP EMBEDDING LIGHTGCN CHO DROPOUTNET]
import os
import torch

os.makedirs("./emb", exist_ok=True)
user_emb_path = "./emb/mooccubex_cold_item_LightGCN_user_emb.pt"
item_emb_path = "./emb/mooccubex_cold_item_LightGCN_item_emb.pt"

# Xây dựng ma trận kề chuẩn hóa A của đồ thị huấn luyện G_train (chỉ chứa các cạnh trên tập khóa ấm Iw)
train_u = torch.tensor(tr_cold['user_id'].values, dtype=torch.long, device=device)
train_i = torch.tensor(tr_cold['course_id'].values, dtype=torch.long, device=device)

num_nodes = n_users + n_items
row_idx = torch.cat([train_u, train_i + n_users])
col_idx = torch.cat([train_i + n_users, train_u])
deg = torch.bincount(row_idx, minlength=num_nodes).float()
deg_inv_sqrt = deg.pow(-0.5)
deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
edge_weights = deg_inv_sqrt[row_idx] * deg_inv_sqrt[col_idx]

adj_indices = torch.stack([row_idx, col_idx], dim=0)
norm_adj = torch.sparse_coo_tensor(adj_indices, edge_weights, (num_nodes, num_nodes)).coalesce().to(device)

# Khởi tạo bảng nhúng định danh theo Xavier Uniform và lan truyền L = 3 tầng LightGCN
torch.manual_seed(42)
E_u0 = nn.Parameter(torch.empty(n_users, 64, device=device))
E_i0 = nn.Parameter(torch.empty(n_items, 64, device=device))
nn.init.xavier_uniform_(E_u0)
nn.init.xavier_uniform_(E_i0)

warm_items_tensor = torch.tensor(warm_item_list, dtype=torch.long, device=device)
cold_items_tensor = torch.tensor(sorted(list(cold_item_set)), dtype=torch.long, device=device)

def lightgcn_propagate(eu, ei, n_layers=3):
    # Đảm bảo khóa lạnh không tham gia truyền thông điệp
    ei_masked = ei.clone()
    ei_masked[cold_items_tensor] = 0.0
    all_emb = torch.cat([eu, ei_masked], dim=0)
    embs = [all_emb]
    for _ in range(n_layers):
        all_emb = torch.sparse.mm(norm_adj, all_emb)
        embs.append(all_emb)
    out_emb = torch.stack(embs, dim=0).mean(dim=0)
    gu, gi = torch.split(out_emb, [n_users, n_items], dim=0)
    gi = gi.clone()
    gi[cold_items_tensor] = 0.0  # Đặt g_i = 0 cho mọi khóa học lạnh i thuộc Ic
    return gu, gi

# Huấn luyện nhanh Pretrain LightGCN với BPR Loss và lấy mẫu âm nghiêm ngặt trên Iw
opt_lgcn = torch.optim.Adam([E_u0, E_i0], lr=0.005)
bpr_bs = 4096
n_edges = len(train_u)

for ep in range(30):
    perm = torch.randperm(n_edges, device=device)
    for s in range(0, n_edges, bpr_bs):
        idx = perm[s:s + bpr_bs]
        u_b = train_u[idx]
        pos_b = train_i[idx]
        # Lấy mẫu âm ngẫu nhiên đều chỉ từ tập khóa học ấm Iw
        neg_b = warm_items_tensor[torch.randint(0, len(warm_items_tensor), (len(idx),), device=device)]
        
        gu, gi = lightgcn_propagate(E_u0, E_i0, n_layers=3)
        pos_scores = (gu[u_b] * gi[pos_b]).sum(dim=-1)
        neg_scores = (gu[u_b] * gi[neg_b]).sum(dim=-1)
        bpr_loss = -F.logsigmoid(pos_scores - neg_scores).mean() + 1e-4 * (
            E_u0[u_b].norm(2).pow(2) + E_i0[pos_b].norm(2).pow(2) + E_i0[neg_b].norm(2).pow(2)
        ) / len(idx)
        
        opt_lgcn.zero_grad()
        bpr_loss.backward()
        opt_lgcn.step()

with torch.no_grad():
    pretrain_gu, pretrain_gi = lightgcn_propagate(E_u0, E_i0, n_layers=3)
    pretrain_gu = pretrain_gu.detach()
    pretrain_gi = pretrain_gi.detach()
    torch.save(pretrain_gu.cpu(), user_emb_path)
    torch.save(pretrain_gi.cpu(), item_emb_path)

print("[Success] Đã huấn luyện và lưu tệp embedding pretrain cho LightGCN (g_i = 0 với mọi khóa lạnh Ic) tại ./emb/")

/tmp/ipykernel_48/2651310416.py:22: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  norm_adj = torch.sparse_coo_tensor(adj_indices, edge_weights, (num_nodes, num_nodes)).coalesce().to(device)


[Success] Đã huấn luyện và lưu tệp embedding pretrain cho LightGCN (g_i = 0 với mọi khóa lạnh Ic) tại ./emb/


In [7]:
# [CELL 6 - KHỞI TẠO CẤU HÌNH THỰC THI CHO DROPOUTNET]
from model.DropoutNet import DropoutNet

class MockArgs:
    def __init__(self, lr, n_dropout, epochs, reg=1e-4):
        self.lr = lr
        self.n_dropout = n_dropout  # Sử dụng đúng tên biến n_dropout theo repo ColdRec
        self.epochs = epochs        
        self.epoch = epochs         
        self.bs = 1024              
        self.batch_size = 1024      
        self.weight_decay = reg
        self.l2 = reg
        self.reg = reg             
        self.topN = "5,10,20"       # Hỗ trợ đầy đủ K ∈ {5, 10, 20}
        self.model = "DropoutNet"   
        self.dataset = "mooccubex"  
        self.emb_size = 64          
        self.early_stop = 10        
        self.early_stop_patience = 10 
        self.cold_object = 'item'   
        self.backbone = "LightGCN"

class MockData:
    def __init__(self, n_users, n_items, sem_tensor):
        self.n_users = n_users
        self.n_items = n_items
        self.mapped_item_content = sem_tensor.cpu().numpy()
        self.item_content_dim = sem_tensor.shape[1]
        self.user_content_dim = 64

class DropoutNetConfig:
    def __init__(self, lr=0.001, n_dropout=0.3, batch_size=1024, epochs=30, reg=1e-4):
        self.lr = lr
        self.dropout = n_dropout
        self.batch_size = batch_size
        self.epochs = epochs
        self.reg = reg
        self.dataset = "mooccubex"
        self.device = str(device)
        self.cuda = torch.cuda.is_available()
        self.args = MockArgs(lr, n_dropout, epochs, reg=reg)
        self.data = MockData(n_users, n_items, sem_tensor)

class DropoutNetModule(nn.Module):
    """Mô-đun chiếu ngữ nghĩa 2 lớp (d_t -> d -> d với GELU) kết hợp ID-Dropout kế thừa từ DropoutNet."""
    def __init__(self, content_dim, emb_dim=64, n_dropout=0.5):
        super().__init__()
        self.n_dropout = n_dropout
        self.proj = nn.Sequential(
            nn.Linear(content_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim)
        )
        for m in self.proj:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_item_train(self, gi_batch, si_batch):
        f_si = self.proj(si_batch)
        if self.n_dropout > 0.0:
            # Mặt nạ Bernoulli độc lập m_i ~ Bernoulli(1 - p) cho từng khóa học trong batch
            keep_mask = (torch.rand(gi_batch.size(0), 1, device=gi_batch.device) >= self.n_dropout).float()
            gi_dropped = keep_mask * gi_batch
        else:
            gi_dropped = gi_batch
        return gi_dropped + f_si

    @torch.no_grad()
    def get_all_item_representations(self, gi_all, si_all, cold_indices):
        f_si = self.proj(si_all)
        hi = gi_all + f_si
        # Suy luận khóa lạnh i thuộc Ic: đặt g_i = 0, tức h_i = f(s_i)
        hi[cold_indices] = f_si[cold_indices]
        return hi

def train_one_epoch_dropoutnet(net, optimizer, train_u, train_i, gu_all, gi_all, si_all, warm_items_t, bs=4096):
    net.train()
    n_edges = len(train_u)
    perm = torch.randperm(n_edges, device=device)
    for s in range(0, n_edges, bs):
        idx = perm[s:s + bs]
        u_b = train_u[idx]
        pos_b = train_i[idx]
        # Lấy mẫu âm nghiêm ngặt chỉ trong tập khóa học ấm Iw
        neg_b = warm_items_t[torch.randint(0, len(warm_items_t), (len(idx),), device=device)]
        
        u_rep = gu_all[u_b]
        pos_h = net.forward_item_train(gi_all[pos_b], si_all[pos_b])
        neg_h = net.forward_item_train(gi_all[neg_b], si_all[neg_b])
        
        pos_sc = (u_rep * pos_h).sum(dim=-1)
        neg_sc = (u_rep * neg_h).sum(dim=-1)
        loss = -F.logsigmoid(pos_sc - neg_sc).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def make_scores_fn(net, gu_all, gi_all, si_all, cold_indices):
    net.eval()
    with torch.no_grad():
        hi_all = net.get_all_item_representations(gi_all, si_all, cold_indices)
    def _scores_fn(u_t):
        u_dev = u_t.to(device)
        return torch.matmul(gu_all[u_dev], hi_all.T)
    return _scores_fn

In [8]:
# [CELL 7 - TUNING 20 TRIALS TRÊN COLD-VAL DÙNG HÀM EVALUATE()]
print("[Info] Bắt đầu quá trình tuning 20 trials trên cold-val (Seed 42)...")

best_val_ndcg = -1.0
best_params = {'lr': 0.001, 'n_dropout': 0.2, 'reg': 1e-4}

lr_candidates = [0.0001, 0.0005, 0.001, 0.005, 0.01]
dropout_candidates = [0.2, 0.5, 0.8]
reg_candidates = [1e-5, 1e-4, 1e-3]

trial_grid = [
    (lr, p_drop, reg)
    for lr in lr_candidates
    for p_drop in dropout_candidates
    for reg in reg_candidates
][:20]

# Chuyển cand_list_va thành tensor boolean mask để tránh lỗi toán tử ~ trên số nguyên của evaluate()
cand_mask_va = torch.zeros(n_items, dtype=torch.bool)
cand_mask_va[torch.as_tensor(cand_list_va)] = True
cand_tensor_va = torch.where(cand_mask_va)[0] # Hoặc truyền thẳng boolean mask nếu sửa nhẹ, tuy nhiên ta dùng index đảo ngược hoặc tensor hợp lệ

for trial_idx, (lr, p_drop, reg) in enumerate(trial_grid, start=1):
    torch.manual_seed(42)
    np.random.seed(42)
    
    config_t = DropoutNetConfig(lr=lr, n_dropout=p_drop, epochs=50, reg=reg)
    model_t = DropoutNet(config_t)
    net_t = DropoutNetModule(sem_tensor.shape[1], emb_dim=64, n_dropout=p_drop).to(device)
    opt_t = torch.optim.Adam(net_t.parameters(), lr=lr, weight_decay=reg)
    
    best_trial_ndcg = -1.0
    patience_t = 10
    patience_cnt = 0
    
    for ep in range(1, 51):
        train_one_epoch_dropoutnet(
            net_t, opt_t, train_u, train_i, pretrain_gu, pretrain_gi, sem_tensor, warm_items_tensor
        )
        if ep % 5 == 0:
            trial_scores_fn = make_scores_fn(net_t, pretrain_gu, pretrain_gi, sem_tensor, cold_items_tensor)
            
            cand_mask_va = torch.zeros(n_items, dtype=torch.bool)
            cand_mask_va[torch.as_tensor(cand_list_va)] = True
            
            # Trong vòng lặp tuning, gọi evaluate với cand là boolean tensor:
            val_res = evaluate(
                trial_scores_fn, target_va, mask_tr, ks=(20,), cand=cand_mask_va, tie_seed=42
            )
            ndcg_cur = float(np.mean(val_res.get('NDCG@20', [0.0])))
            if ndcg_cur > best_trial_ndcg:
                best_trial_ndcg = ndcg_cur
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= patience_t:
                    break
        
    print(f"  [Trial {trial_idx}/20] lr={lr}, n_dropout={p_drop}, reg={reg} -> Val NDCG@20 (Cold-Val): {best_trial_ndcg:.4f}")
    
    if best_trial_ndcg > best_val_ndcg:
        best_val_ndcg = best_trial_ndcg
        best_params = {'lr': lr, 'n_dropout': p_drop, 'reg': reg}

print(f"\n[Info] Hoàn thành tuning! Cấu hình tối ưu: {best_params} (NDCG@20 cao nhất: {best_val_ndcg:.4f})")

[Info] Bắt đầu quá trình tuning 20 trials trên cold-val (Seed 42)...
  [Trial 1/20] lr=0.0001, n_dropout=0.2, reg=1e-05 -> Val NDCG@20 (Cold-Val): 0.0568
  [Trial 2/20] lr=0.0001, n_dropout=0.2, reg=0.0001 -> Val NDCG@20 (Cold-Val): 0.0527
  [Trial 3/20] lr=0.0001, n_dropout=0.2, reg=0.001 -> Val NDCG@20 (Cold-Val): 0.0472
  [Trial 4/20] lr=0.0001, n_dropout=0.5, reg=1e-05 -> Val NDCG@20 (Cold-Val): 0.0622
  [Trial 5/20] lr=0.0001, n_dropout=0.5, reg=0.0001 -> Val NDCG@20 (Cold-Val): 0.0600
  [Trial 6/20] lr=0.0001, n_dropout=0.5, reg=0.001 -> Val NDCG@20 (Cold-Val): 0.0484
  [Trial 7/20] lr=0.0001, n_dropout=0.8, reg=1e-05 -> Val NDCG@20 (Cold-Val): 0.0659
  [Trial 8/20] lr=0.0001, n_dropout=0.8, reg=0.0001 -> Val NDCG@20 (Cold-Val): 0.0649
  [Trial 9/20] lr=0.0001, n_dropout=0.8, reg=0.001 -> Val NDCG@20 (Cold-Val): 0.0547
  [Trial 10/20] lr=0.0005, n_dropout=0.2, reg=1e-05 -> Val NDCG@20 (Cold-Val): 0.0743
  [Trial 11/20] lr=0.0005, n_dropout=0.2, reg=0.0001 -> Val NDCG@20 (Cold-Val

In [9]:
# [CELL 8 - CHẠY 5 SEEDS CHÍNH THỨC VỚI HÀM EVALUATE() NGUYÊN BẢN]
import copy

SEEDS = [42, 2023, 2024, 1007, 999]
final_results_a = []
final_results_b = []
epoch_times = []
latencies = []

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# Tổng hợp mask kết hợp train và val để tránh rò rỉ dữ liệu khi test
mask_test_combined = {u: mask_tr.get(u, set()) | mask_va.get(u, set()) for u in target_te.keys()}
# Tạo boolean mask cho cold-test items (Tập a)
cand_mask_te = torch.zeros(n_items, dtype=torch.bool)
cand_mask_te[torch.as_tensor(cand_list_te)] = True

print(f"\n[Info] Bắt đầu chạy chính thức qua 5 seeds với best_params: {best_params}")
for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    config_seed = DropoutNetConfig(
        lr=best_params['lr'], n_dropout=best_params['n_dropout'], epochs=300, reg=best_params.get('reg', 1e-4)
    )
    model = DropoutNet(config_seed)
    net = DropoutNetModule(sem_tensor.shape[1], emb_dim=64, n_dropout=best_params['n_dropout']).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=best_params['lr'], weight_decay=best_params.get('reg', 1e-4))
    
    start_ep = time.time()
    patience = 10
    patience_counter = 0
    best_seed_ndcg = -1.0
    best_state = copy.deepcopy(net.state_dict())
    
    actual_epochs = 300
    for epoch in range(1, 301):
        train_one_epoch_dropoutnet(
            net, optimizer, train_u, train_i, pretrain_gu, pretrain_gi, sem_tensor, warm_items_tensor
        )
        if epoch % 5 == 0:
            val_scores_fn = make_scores_fn(net, pretrain_gu, pretrain_gi, sem_tensor, cold_items_tensor)
            val_res = evaluate(
                val_scores_fn, target_va, mask_tr, ks=(20,), cand=torch.as_tensor(cand_list_va), tie_seed=seed
            )
            cur_ndcg = float(np.mean(val_res.get('NDCG@20', [0.0])) )
            if cur_ndcg > best_seed_ndcg:
                best_seed_ndcg = cur_ndcg
                best_state = copy.deepcopy(net.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    actual_epochs = epoch
                    break
                
    ep_duration = (time.time() - start_ep) / max(1, actual_epochs)
    epoch_times.append(ep_duration)
    
    net.load_state_dict(best_state)
    
    start_inf = time.time()
    test_scores_fn = make_scores_fn(net, pretrain_gu, pretrain_gi, sem_tensor, cold_items_tensor)
        
    # Đánh giá tập ứng viên (a): Chỉ trên các khóa lạnh test
    res_a = evaluate(
        test_scores_fn, target_te, mask_test_combined, ks=(5, 10, 20), cand=cand_mask_te, tie_seed=seed
    )
    
    # Đánh giá tập ứng viên (b): Toàn bộ danh mục khóa học (All Items -> cand=None)
    res_b = evaluate(
        test_scores_fn, target_te, mask_test_combined, ks=(5, 10, 20), cand=None, tie_seed=seed
    )
    
    end_inf = time.time()
    latencies.append(((end_inf - start_inf) * 1000) / max(1, len(target_te)))
    
    final_results_a.append(res_a)
    final_results_b.append(res_b)
    
    ra20 = float(np.mean(res_a.get('Recall@20', [0.0])))
    na20 = float(np.mean(res_a.get('NDCG@20', [0.0])))
    print(f"-> Hoàn thành Seed {seed} (Epochs: {actual_epochs}, Best Val NDCG@20: {best_seed_ndcg:.4f}) | [Tập a] Recall@20: {ra20:.4f}, NDCG@20: {na20:.4f}")

mean_vram = get_vram_usage()
mean_lat = np.mean(latencies)
mean_ep_time = np.mean(epoch_times)

print("\n" + "="*85)
print(" KẾT QUẢ THỰC NGHIỆM CHÍNH THỨC - DROPOUTNET (5 SEEDS - HÀM EVALUATE GỐC)")
print("="*85)
print(f" [Hiệu năng] Thời gian: {mean_ep_time:.4f} s/epoch | VRAM: {mean_vram:.2f} GB | Latency: {mean_lat:.4f} ms/user")

print("\n--- TẬP ỨNG VIÊN (A): CHỈ XẾP HẠNG TRONG CÁC KHÓA LẠNH ---")
for k in [5, 10, 20]:
    r_a = np.mean([np.mean(r.get(f'Recall@{k}', [0.0])) for r in final_results_a])
    s_ra = np.std([np.mean(r.get(f'Recall@{k}', [0.0])) for r in final_results_a])
    n_a = np.mean([np.mean(r.get(f'NDCG@{k}', [0.0])) for r in final_results_a])
    s_na = np.std([np.mean(r.get(f'NDCG@{k}', [0.0])) for r in final_results_a])
    hr_a = np.mean([np.mean(r.get(f'HR@{k}', [0.0])) for r in final_results_a])
    print(f" Recall@{k}: {r_a:.4f} ± {s_ra:.4f} | NDCG@{k}: {n_a:.4f} ± {s_na:.4f} | HR@{k}: {hr_a:.4f}")

print("\n--- TẬP ỨNG VIÊN (B): XẾP HẠNG TRÊN TOÀN BỘ DANH MỤC KHÓA HỌC (ALL ITEMS) ---")
for k in [5, 10, 20]:
    r_b = np.mean([np.mean(r.get(f'Recall@{k}', [0.0])) for r in final_results_b])
    s_rb = np.std([np.mean(r.get(f'Recall@{k}', [0.0])) for r in final_results_b])
    n_b = np.mean([np.mean(r.get(f'NDCG@ {k}', [0.0])) for r in final_results_b])
    s_nb = np.std([np.mean(r.get(f'NDCG@{k}', [0.0])) for r in final_results_b])
    hr_b = np.mean([np.mean(r.get(f'HR@{k}', [0.0])) for r in final_results_b])
    print(f" Recall@{k}: {r_b:.4f} ± {s_rb:.4f} | NDCG@{k}: {n_b:.4f} ± {s_nb:.4f} | HR@{k}: {hr_b:.4f}")
print("="*85)


[Info] Bắt đầu chạy chính thức qua 5 seeds với best_params: {'lr': 0.0005, 'n_dropout': 0.8, 'reg': 1e-05}
-> Hoàn thành Seed 42 (Epochs: 185, Best Val NDCG@20: 0.0002) | [Tập a] Recall@20: 0.1463, NDCG@20: 0.0695
-> Hoàn thành Seed 2023 (Epochs: 55, Best Val NDCG@20: 0.0000) | [Tập a] Recall@20: 0.1752, NDCG@20: 0.0744
-> Hoàn thành Seed 2024 (Epochs: 185, Best Val NDCG@20: 0.0000) | [Tập a] Recall@20: 0.1500, NDCG@20: 0.0725
-> Hoàn thành Seed 1007 (Epochs: 55, Best Val NDCG@20: 0.0000) | [Tập a] Recall@20: 0.1640, NDCG@20: 0.0698
-> Hoàn thành Seed 999 (Epochs: 105, Best Val NDCG@20: 0.0000) | [Tập a] Recall@20: 0.1710, NDCG@20: 0.0760

 KẾT QUẢ THỰC NGHIỆM CHÍNH THỨC - DROPOUTNET (5 SEEDS - HÀM EVALUATE GỐC)
 [Hiệu năng] Thời gian: 0.3523 s/epoch | VRAM: 0.26 GB | Latency: 0.7317 ms/user

--- TẬP ỨNG VIÊN (A): CHỈ XẾP HẠNG TRONG CÁC KHÓA LẠNH ---
 Recall@5: 0.0644 ± 0.0054 | NDCG@5: 0.0439 ± 0.0025 | HR@5: 0.0790
 Recall@10: 0.1014 ± 0.0096 | NDCG@10: 0.0564 ± 0.0030 | HR@10: 0.12